# 04.06.02 Image Forensic Analysis

**Phase 04 — Document Processing and OCR | Module 04.06.02**  
Ophthalmological Foundation of the Philippines (OFPHIL) Research Pipeline

### Research Objectives
1. Implement physics-based and statistical forensic screening on document images.
2. Perform Error Level Analysis (ELA) using JPEG recompression quantization deltas.
3. Compute high-frequency local noise residual variance via Laplacian kernel estimation.
4. Extract EXIF and compression metadata indicators without relying on learned deep models.
5. Export image forensic indicator matrices for downstream classifier benchmarking.


### Inputs & Expected Artifacts
- **Input Manifest:** `data/processed/tampering/splits/test.csv` (or pilot document inventory).
- **Output Directory:** `data/interim/document_processing/forensics/`.
- **Forensic Features:** `data/interim/document_processing/forensics/forensic_indicators_report.csv`.


### 1. Environment & Path Configuration


In [2]:
import os
import sys
import cv2
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

def _find_root(p=Path.cwd()):
    for parent in [p, *p.parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

OUT_FORENSICS_DIR = paths.interim_doc_proc / 'forensics'
OUT_FORENSICS_DIR.mkdir(parents=True, exist_ok=True)
TEST_MANIFEST = paths.tampering_dir / 'splits' / 'test.csv'

print(f"Project Root           : {paths.root}")
print(f"Forensic Output Dir    : {OUT_FORENSICS_DIR}")
print(f"Test Manifest Path     : {TEST_MANIFEST} (Exists: {TEST_MANIFEST.exists()})")


Project Root           : C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2
Forensic Output Dir    : C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\document_processing\forensics
Test Manifest Path     : C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\processed\tampering\splits\test.csv (Exists: True)


### 2. Error Level Analysis (ELA) Implementation


In [3]:
# Error Level Analysis (ELA) Engine
def compute_ela(image_path: Path, quality: int = 90, scale: int = 15) -> np.ndarray:
    """Computes Error Level Analysis (ELA) pixel variance via JPEG recompression."""
    orig = Image.open(image_path).convert('RGB')
    temp_path = OUT_FORENSICS_DIR / '_tmp_ela.jpg'
    orig.save(temp_path, 'JPEG', quality=quality)
    resaved = Image.open(temp_path)
    diff = np.abs(np.array(orig, dtype=np.float32) - np.array(resaved, dtype=np.float32))
    if temp_path.exists():
        temp_path.unlink()
    ela_map = np.clip(diff * scale, 0, 255).astype(np.uint8)
    return ela_map

print("ELA forensic engine initialized with temporary file cleanup.")


ELA forensic engine initialized with temporary file cleanup.


### 3. Local Noise Variance Estimation


In [4]:
# High-frequency local noise residual estimation via Laplacian filtering
def compute_noise_variance(img_gray: np.ndarray, ksize: int = 7) -> float:
    lap = cv2.Laplacian(img_gray, cv2.CV_64F, ksize=ksize)
    return float(np.var(lap))

print("Laplacian noise variance metric defined.")


Laplacian noise variance metric defined.


### 4. Forensic Indicator Extraction Batch Pipeline


In [5]:
# Execute forensic screening across test split samples
forensic_records = []
target_out_csv = OUT_FORENSICS_DIR / 'forensic_indicators_report.csv'

manifest_to_use = TEST_MANIFEST if TEST_MANIFEST.exists() else paths.doc_inventory_manifest
if manifest_to_use.exists():
    df_samples = pd.read_csv(manifest_to_use)
    print(f"Processing forensic indicators across {len(df_samples)} candidate images...")
    for idx, row in df_samples.head(50).iterrows():
        rel_p = row.get('relative_path') if 'relative_path' in row else row.get('file_name')
        img_p = paths.root / rel_p if rel_p else None
        if img_p and img_p.exists():
            img_bgr = cv2.imread(str(img_p))
            if img_bgr is not None:
                gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
                noise_var = compute_noise_variance(gray)
                ela_arr = compute_ela(img_p)
                ela_mean = float(np.mean(ela_arr))
                forensic_records.append({
                    'file_name': img_p.name,
                    'relative_path': str(img_p.relative_to(paths.root)).replace('\\', '/'),
                    'noise_variance': round(noise_var, 2),
                    'ela_mean_intensity': round(ela_mean, 2),
                    'status': 'SCREENED'
                })
    df_forensics = pd.DataFrame(forensic_records)
    df_forensics.to_csv(target_out_csv, index=False)
    print(f"Successfully exported {len(df_forensics)} forensic indicators -> {target_out_csv}")
else:
    print(f"Status: READY_FOR_EXECUTION (Awaiting input manifest at {manifest_to_use}).")


Processing forensic indicators across 341 candidate images...
Successfully exported 50 forensic indicators -> C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\document_processing\forensics\forensic_indicators_report.csv


### Summary & Transition
- Forensic features (ELA intensity, noise residual variance) extracted without deep model overhead.
- Features ready for Module 04.06.03 Tampering Detection Model Benchmark.
